## Architecture reference for this lab

**Step 19 — SDK / Notebook Edition (full picture)**

![Step 19 — SDK / Notebook Edition (full picture)](images/step-19-sdk-notebook-edition.png)



# Lab 8 — Cleanup (remove only the `-sdk` resources)

**What this lab is.** This deletes everything the SDK build created, so you stop paying for
anything you no longer need. It carefully removes **only** the resources with the `-sdk`
suffix.

**Why we do it.** Cloud resources cost money while they exist. Once you're done, leaving the
Knowledge Base, runtime, gateway, database, etc. running would keep charging you. This lab
tears them down cleanly and in the right order.

**Why it's needed here.** Some resources keep billing until explicitly deleted (a vector
index, a deployed runtime). Deleting in the correct order avoids "can't delete, still in use"
errors and leftover charges.

**How it helps the project.** It closes the loop responsibly: build, test, then clean up.

**What it deliberately keeps:** your shared `careconnect-approved-docs` bucket (used by your
console build too) and anything without the `-sdk` suffix — i.e. it never touches your
original build.

> **Also remember:** this build ran in **SageMaker**, so when you're completely finished, stop
> or delete the SageMaker space/app as well, or the notebook compute keeps costing money.

---

## Prerequisite — run these two setup cells first (every lab has them)

Before this lab's own steps, run the **two setup cells** below. Every notebook (lab-00
through lab-08) starts with these same two cells — on purpose, not by mistake.

**Why they repeat in every notebook.** Each notebook runs in its own fresh "kernel"
(a separate Python session) with no memory of the other notebooks. So each one has to set
itself up from scratch. These two cells are that setup.

**Cell 1 — "bootstrap":** finds the project's main folder (the one containing `lab_helpers`)
no matter where the notebook is opened from, so `import lab_helpers...` always works.

**Cell 2 — "preflight":** checks all helper files are present before the lab begins, and stops
with a clear message if anything is missing — instead of failing confusingly later.

**Do I run them?** Yes — run both, in order, at the top of **every** lab. They take a second
and prevent the most common setup problems. After these two, continue with the lab's steps.

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - It looks at the current folder, then its parent, then its parent's parent, and so on,
#   until it finds the folder that contains "lab_helpers". That folder is our project root.
# - It then switches into that folder and adds it to Python's search path, so that
#   'import lab_helpers...' works from anywhere.
# - If it never finds "lab_helpers", it stops with a clear message instead of a confusing error later.
# You do not need to edit anything here — just run it first.
# === CareConnect bootstrap — run me first ===
# Makes this notebook work from any folder and gives a clear error if the
# lab_helpers package is missing (e.g. not uploaded to the repo).
import os, sys

def _find_repo_root(start=None):
    here = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(here, "lab_helpers")):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            return None
        here = parent

_root = _find_repo_root()
if _root is None:
    raise RuntimeError(
        "Could not find the 'lab_helpers/' folder from " + os.getcwd() + ".\n"
        "This means the helper package is not next to the notebooks.\n"
        "Fix: make sure lab_helpers/ and requirements.txt are in the same folder\n"
        "as these .ipynb files (see README > Setup). Then re-run this cell.")
os.chdir(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
print("Repo root:", _root)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


In [ ]:
# WHAT THIS CELL DOES (plain English):
# - It checks that each helper file we rely on is actually present on disk.
# - If any are missing, it stops now with a clear list of what to upload, rather than
#   failing in the middle of the lab.
# Run this straight after the bootstrap cell above.
# === Preflight: confirm every helper file is present BEFORE running the lab ===
import os
_required = [
    "requirements.txt",
    "lab_helpers/__init__.py",
    "lab_helpers/utils.py",
    "lab_helpers/careconnect_agents.py",
    "lab_helpers/deterministic_safety.py",
    "lab_helpers/runtime_entrypoint.py",
]
_missing = [f for f in _required if not os.path.isfile(f)]
if _missing:
    raise RuntimeError("Missing required files:\n  - " + "\n  - ".join(_missing) +
        "\n\nUpload the full lab_helpers/ folder + requirements.txt, then re-run.")
print("Preflight OK — all helper files present.")

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 1: Delete the AgentCore Runtime + ECR image

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes the deployed Supervisor (the AgentCore Runtime). 'try/except' means: if it's
#   already gone, just skip it without error.
import boto3
import lab_helpers.utils as u
acc = boto3.client("bedrock-agentcore-control", region_name=u.REGION)
try:
    arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/runtime_arn")
    rid = arn.split("/")[-1]
    acc.delete_agent_runtime(agentRuntimeId=rid)
    print("Deleted runtime", rid)
except Exception as e:
    print("Runtime delete skipped:", e)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 2: Delete API Gateway + proxy Lambda

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes the public API and the two Lambda functions (proxy + mock tools) we created.
apigw = boto3.client("apigateway", region_name=u.REGION)
lam = boto3.client("lambda", region_name=u.REGION)
for api in apigw.get_rest_apis().get("items", []):
    if api["name"] == u.name("careconnect-api"):
        apigw.delete_rest_api(restApiId=api["id"]); print("Deleted API", api["id"])
for fn in (u.PROXY_LAMBDA, u.MOCK_TOOLS_LAMBDA):
    try: lam.delete_function(FunctionName=fn); print("Deleted Lambda", fn)
    except Exception as e: print(fn, "skip:", e)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 3: Delete Gateway + target

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes the Gateway and its attached tool target.
try:
    gid = u.get_ssm_parameter(f"{u.SSM_PREFIX}/gateway_id")
    for t in acc.list_gateway_targets(gatewayIdentifier=gid).get("items", []):
        acc.delete_gateway_target(gatewayIdentifier=gid, targetId=t["targetId"])
    acc.delete_gateway(gatewayIdentifier=gid); print("Deleted gateway", gid)
except Exception as e:
    print("Gateway delete skipped:", e)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 4: Delete Knowledge Base + confirm the S3 Vectors index is gone

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes the Knowledge Base and its data source.
# - IMPORTANT: it reminds you to check the S3 Vectors console and confirm the vector
#   index/bucket is also gone, because that can bill separately if left behind.
ba = boto3.client("bedrock-agent", region_name=u.REGION)
try:
    kb_id = u.get_ssm_parameter(f"{u.SSM_PREFIX}/kb_id")
    for ds in ba.list_data_sources(knowledgeBaseId=kb_id).get("dataSourceSummaries", []):
        ba.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds["dataSourceId"])
    ba.delete_knowledge_base(knowledgeBaseId=kb_id)
    print("Deleted KB", kb_id)
    print("NOW verify in S3 Vectors console that the vector index/bucket for the KB is gone.")
except Exception as e:
    print("KB delete skipped:", e)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 5: Delete DynamoDB table + Step Functions state machine

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes the escalation tickets table and the approval workflow (Step Functions).
ddb = boto3.client("dynamodb", region_name=u.REGION)
sfn = boto3.client("stepfunctions", region_name=u.REGION)
try: ddb.delete_table(TableName=u.ESCALATION_TABLE); print("Deleted table")
except Exception as e: print("table skip:", e)
try:
    arn = u.get_ssm_parameter(f"{u.SSM_PREFIX}/state_machine_arn")
    sfn.delete_state_machine(stateMachineArn=arn); print("Deleted state machine")
except Exception as e: print("sfn skip:", e)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 6: Delete the Guardrail

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes the safety Guardrail we created.
bedrock = boto3.client("bedrock", region_name=u.REGION)
try:
    bedrock.delete_guardrail(guardrailIdentifier=u.get_ssm_parameter(f"{u.SSM_PREFIX}/guardrail_id"))
    print("Deleted guardrail")
except Exception as e: print("guardrail skip:", e)

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


### Step 7: Delete the `-sdk` IAM roles and SSM parameters

In [ ]:
# WHAT THIS CELL DOES (plain English):
# - Deletes all the -sdk IAM permission roles we made, and clears the values we saved in
#   Parameter Store. After this, the SDK build is fully cleaned up.
iam = boto3.client("iam")
roles = [u.name(r) for r in ["CareConnectKBRole","CareConnectRuntimeRole",
    "CareConnectGatewayRole","CareConnectMockToolsRole","CareConnectProxyRole",
    "CareConnectSfnRole"]]
for r in roles:
    try:
        for p in iam.list_role_policies(RoleName=r)["PolicyNames"]:
            iam.delete_role_policy(RoleName=r, PolicyName=p)
        iam.delete_role(RoleName=r); print("Deleted role", r)
    except Exception as e: print(r, "skip:", e)

ssm = boto3.client("ssm")
for name in ["kb_id","guardrail_id","guardrail_version","gateway_id","gateway_url",
             "state_machine_arn","runtime_arn","api_url"]:
    u.delete_ssm_parameter(f"{u.SSM_PREFIX}/{name}")
print("Cleared SSM parameters.")

# (This cell produces no visible output when it succeeds — it defines/creates things silently.)


## Cleanup complete ✅

Removed only `-sdk` resources. The shared `careconnect-approved-docs` bucket and your
original console/CLI build are untouched.

**Not deleted on purpose:** the reused documents bucket, and (if you created one) any
customer-managed KMS key. Also remember: this SDK build ran in **SageMaker**, so when you
are fully done, stop/delete the SageMaker Studio space/app to stop notebook compute cost.